### Converting .flv files to .mp4s and applying OpenFace

In [ ]:
import os               # VSCode Navigator
import subprocess       # Terminal Access
import pandas as pd

video_dir = '/Users/songye/Desktop/Dev/REU/CREMA-D/VideoFlash'
mp4_dir = '/Users/songye/Desktop/Dev/REU/CREMA-D/mp4'
output_dir = '/Users/songye/Desktop/Dev/REU/CREMA-D/output'


os.makedirs(mp4_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)


# A list of all the files
flv_video_files = [video for video in os.listdir(video_dir) if video.endswith('.flv')][:2500]

for video in flv_video_files:

    # This converts the .flv file into mp4 files for OpenFace
    mp4_video_name = video.replace(".flv", ".mp4")
    subprocess.run([
        'ffmpeg',
        '-i',
        os.path.join(video_dir, video),
        '-y',
        '-loglevel',
        'error',
        os.path.join(mp4_dir, mp4_video_name)])
    

    # subprocess.run([
    # "docker", "run", "--rm",
    # "--platform", "linux/amd64",  # explicitly force amd64 emulation
    # "-v", "/Users/songye/Desktop/Dev/REU/CREMA-D:/data",
    # "algebr/openface:latest",
    # "/home/openface-build/build/bin/FeatureExtraction",
    # "-f", "/data/mp4/" + mp4_video_name,
    # "-out_dir", "/data/output/"])

    # This did not work, so have to run Docker manually via Terminal
    # docker run -it -v /Users/songye/Desktop/Dev/REU/CREMA-D:/data algebr/openface:latest
    # for f in /data/mp4/*.mp4; do /home/openface-build/build/bin/FeatureExtraction -f "$f" -out_dir /data/output/; done

    

### Constructing the Feature Matrix

In [63]:
# Now we need a feature matrix
# Each row = one entire video, columns = aggregated AU statistics + emotion label (Each AU's mean and std)
list_of_csvs = [csv for csv in os.listdir(output_dir) if csv.endswith(".csv")]
feature_matrix = pd.DataFrame()

count = 0
cols_order = []
#emotion_count = {}

for csv in list_of_csvs:
    df = pd.read_csv(os.path.join(output_dir, csv))
    df.columns = df.columns.str.strip()

    list_means = []
    list_std = []
    list_max = []

    for col in df.columns:
        if col.startswith('AU') and (col.endswith('_r') or col.endswith('_c')):
            list_means.append(df[col].mean())
            list_std.append(df[col].std())
            list_max.append(df[col].max())
            if count == 0:
                cols_order.append(col)

    total_list = list_means.copy()
    total_list.extend(list_std)
    total_list.extend(list_max)

    # To get the emotion from the file name
    emotion = csv.split('_')[2]
    total_list.append(emotion)

    feature_matrix = pd.concat([feature_matrix, pd.DataFrame([total_list])], ignore_index=True)
    count += 1

col_names = ([col + '_mean' for col in cols_order] + 
             [col + '_std' for col in cols_order] + 
             [col + '_max' for col in cols_order] +
             ['Emotion'])

feature_matrix.columns = col_names

print(feature_matrix.head(5))

   AU01_r_mean  AU02_r_mean  AU04_r_mean  AU05_r_mean  AU06_r_mean  \
0     0.056716     0.020597     1.141493     0.024179     0.333881   
1     0.069759     0.038313     0.000361     0.023133     0.015783   
2     0.158750     0.077031     0.076562     0.034844     0.078750   
3     0.154730     0.079865     0.682027     0.028919     0.245405   
4     0.208554     0.156988     1.383855     0.045181     2.165060   

   AU07_r_mean  AU09_r_mean  AU10_r_mean  AU12_r_mean  AU14_r_mean  ...  \
0     0.224925     0.023134     0.534478     0.012090     0.568358  ...   
1     0.000000     0.050361     0.200000     0.132530     0.134458  ...   
2     0.297344     0.107344     0.234844     0.327969     0.437500  ...   
3     0.841622     0.037703     0.453784     0.800270     0.442973  ...   
4     3.725422     0.042892     1.426627     0.623373     1.454940  ...   

   AU14_c_max  AU15_c_max  AU17_c_max  AU20_c_max  AU23_c_max  AU25_c_max  \
0         1.0         0.0         1.0         0.0  

# Model(s) Testing and Training

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

X = feature_matrix.drop(columns = ["Emotion"])
y = feature_matrix["Emotion"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

classifier = SVC(kernel = 'rbf', C = 0.8, gamma = 'scale', class_weight = 'balanced')

classifier.fit(X_train_scaled, y_train)
y_pred = classifier.predict(X_test_scaled)

print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

0.6005747126436781
              precision    recall  f1-score   support

         ANG       0.63      0.56      0.59        61
         DIS       0.71      0.74      0.72        65
         FEA       0.41      0.43      0.42        60
         HAP       0.85      0.93      0.88        54
         NEU       0.53      0.57      0.55        47
         SAD       0.45      0.39      0.42        61

    accuracy                           0.60       348
   macro avg       0.60      0.60      0.60       348
weighted avg       0.60      0.60      0.60       348



In [ ]:
# To find best parameters
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': [1, 0.1, 0.01, 0.001],
    'kernel': ['rbf', 'linear']
}
grid_search = GridSearchCV(estimator=SVC(kernel='rbf'), param_grid=param_grid, cv=5, scoring='accuracy', verbose=1, n_jobs=-1)

grid_search.fit(X_train_scaled, y_train)
y_pred_gs = grid_search.predict(X_test_scaled)
print(accuracy_score(y_test, y_pred_gs))

print("Best Paramaters: ", grid_search.best_params_)

Fitting 5 folds for each of 32 candidates, totalling 160 fits
0.6091954022988506
Best Paramaters:  {'C': 1, 'gamma': 0.01, 'kernel': 'rbf'}
